# 00 — MD fundamentals: integrators and thermal initialization

**Learning goals**

- see how an integrator turns Newton's equation into discrete updates,
- compare explicit Euler, Leapfrog and Velocity Verlet on a harmonic oscillator,
- draw Maxwell–Boltzmann velocities, remove centre-of-mass motion and measure the kinetic temperature,
- understand what an initial velocity rescaling does — and what it removes.

All quantities are in reduced units with $k_B = 1$.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import inspect
import numpy as np
import matplotlib.pyplot as plt

from src import integrators
from src.potentials import harmonic_force, harmonic_potential
from src.thermal import (maxwell_boltzmann_velocities, total_momentum, remove_com_velocity,
                         kinetic_energy, degrees_of_freedom, instantaneous_temperature,
                         rescale_to_temperature)
from src.plotting import set_style, save_figure, PALETTE, REFERENCE
set_style()

## Parameters

Every notebook has one parameter cell like this. Change values and re-run the notebook from here.

In [ ]:
# --- System A: 1D harmonic oscillator -------------------------------------
k, m = 1.0, 1.0                       # spring constant and mass
x0, v0 = 1.0, 0.0                     # initial position and velocity
dt = 0.10                             # timestep (omega * dt = 0.1 here)
t_max = 30.0                          # about five oscillation periods
dt_values = [0.20, 0.10, 0.05, 0.02]  # optional timestep experiment

# --- System B: 2D particles for thermal initialization -------------------
N, dim = 36, 2
mass = 1.0
T_target = 1.0
seed = 2026
rng = np.random.default_rng(seed)

## 1. Newtonian dynamics

MD integrates Newton's equation of motion

$$m\ddot{x} = F(x), \qquad F(x) = -\frac{dV}{dx}.$$

For the harmonic oscillator $V(x)=\tfrac12 kx^2$, $F(x)=-kx$, the exact solution is
$x(t)=x_0\cos\omega t + (v_0/\omega)\sin\omega t$ with $\omega=\sqrt{k/m}$, and the total energy
$E=\tfrac12 mv^2+\tfrac12 kx^2$ is exactly conserved. A numerical integrator replaces the continuous
flow by steps of size $\Delta t$; the question is how faithfully it does so.

In [ ]:
force = lambda x: harmonic_force(x, k)
energy = lambda x, v: 0.5 * m * v**2 + harmonic_potential(x, k)

omega = np.sqrt(k / m)
n_steps = int(round(t_max / dt))
t = np.arange(n_steps + 1) * dt
x_exact = x0 * np.cos(omega * t) + v0 / omega * np.sin(omega * t)
E0 = energy(x0, v0)
print(f"period = {2 * np.pi / omega:.2f}, omega*dt = {omega * dt:.2f}, n_steps = {n_steps}")

## 2. Explicit Euler

$$x_{n+1} = x_n + v_n\Delta t, \qquad v_{n+1} = v_n + a_n\Delta t, \qquad a_n = F(x_n)/m.$$

Euler is the simplest possible scheme, but it is not time-reversible and does not preserve phase-space
volume. For long Hamiltonian trajectories this shows up as a systematic energy drift.

In [ ]:
print(inspect.getsource(integrators.euler))

## 3. Leapfrog

$$v_{n+1/2} = v_{n-1/2} + a_n\Delta t, \qquad x_{n+1} = x_n + v_{n+1/2}\Delta t.$$

Positions live on integer steps and velocities on half steps: the two are staggered by $\Delta t/2$.
To compute an energy at step $n$ we need an on-step velocity, here $v_n = (v_{n-1/2}+v_{n+1/2})/2$.

In [ ]:
print(inspect.getsource(integrators.leapfrog))

## 4. Velocity Verlet

$$x_{n+1} = x_n + v_n\Delta t + \tfrac12 a_n\Delta t^2,$$
then recompute the force $a_{n+1}=F(x_{n+1})/m$, and
$$v_{n+1} = v_n + \tfrac12(a_n + a_{n+1})\Delta t.$$

Positions and velocities are available at the same time, and only one new force evaluation is needed per step.

In [ ]:
print(inspect.getsource(integrators.velocity_verlet_step))

## 5. Integrator comparison

In [ ]:
runs = {
    "Euler": integrators.euler(x0, v0, force, m, dt, n_steps),
    "Leapfrog": integrators.leapfrog(x0, v0, force, m, dt, n_steps)[:2],
    "Velocity Verlet": integrators.velocity_verlet(x0, v0, force, m, dt, n_steps),
}

fig, axes = plt.subplots(3, 1, figsize=(7, 7.2), sharex=True)
axes[0].plot(t, x_exact, ls="--", color=REFERENCE, lw=1.2, label="exact")
for (name, (x, v)), ls in zip(runs.items(), ["-", "-", ":"]):
    E = energy(x, v)
    axes[0].plot(t, x, ls=ls, label=name)
    axes[1].plot(t, E, ls=ls, label=name)
    axes[2].semilogy(t[1:], np.maximum(np.abs(E[1:] - E0) / E0, 1e-16), ls=ls, label=name)
axes[0].set_ylabel(r"$x(t)$")
axes[1].set_ylabel(r"total energy $E$")
axes[2].set_ylabel(r"$|E - E_0| / E_0$")
axes[2].set_xlabel(r"time $t$ (reduced units)")
axes[0].legend(ncol=4, loc="upper left")
axes[0].set_title(rf"harmonic oscillator, $\omega\Delta t$ = {omega * dt:.2f}")
fig.tight_layout()
save_figure(fig, "00_integrator_comparison")
plt.show()

for name, (x, v) in runs.items():
    E = energy(x, v)
    print(f"{name:16s} max |dE|/E0 = {np.max(np.abs(E - E0)) / E0:.2e}   final E/E0 = {E[-1] / E0:.3f}")

**Interpretation.**

- Euler multiplies the oscillator energy by $(1+\omega^2\Delta t^2)$ every step, so the error grows steadily.
- Leapfrog and Velocity Verlet keep the energy error bounded and oscillating, without systematic drift, for this
  stable timestep.
- The Leapfrog and Velocity Verlet curves lie exactly on top of each other: the two schemes generate identical
  positions and differ only in *when* velocities are stored (the dotted line is drawn over the solid one).

Bounded energy error is a property of these time-reversible, symplectic schemes at stable timesteps; it does not
make them the right choice for every problem.

## 6. Optional: timestep experiment

The maximum energy error over the same physical time for several timesteps.

In [ ]:
print(f"{'dt':>6} {'Euler':>12} {'Velocity Verlet':>17}    (max |dE|/E0 up to t = {t_max})")
for dt_i in dt_values:
    n_i = int(round(t_max / dt_i))
    errors = []
    for method in (integrators.euler, integrators.velocity_verlet):
        x, v = method(x0, v0, force, m, dt_i, n_i)
        errors.append(np.max(np.abs(energy(x, v) - E0)) / E0)
    print(f"{dt_i:6.2f} {errors[0]:12.3e} {errors[1]:17.3e}")

Halving $\Delta t$ reduces the Velocity Verlet error by roughly a factor of four (second order), whereas the Euler
error still grows with simulated time. Velocity Verlet is unstable for $\omega\Delta t > 2$; in real MD the fastest
motions (often bond vibrations involving hydrogen) limit the usable timestep.

## 7. Maxwell–Boltzmann velocities

We now switch to a small 2D system of $N$ particles (positions are not needed yet). In the canonical ensemble each
Cartesian velocity component is Gaussian:

$$v_{i,\alpha} \sim \mathcal{N}\!\left(0, \frac{k_BT}{m_i}\right), \qquad \sigma_{v,i} = \sqrt{\frac{k_BT}{m_i}}.$$

In [ ]:
v = maxwell_boltzmann_velocities(mass, T_target, N, dim, rng)
sigma_v = np.sqrt(T_target / mass)

pooled = maxwell_boltzmann_velocities(mass, T_target, 500 * N, dim, rng)  # many independent draws
grid = np.linspace(-4, 4, 200) * sigma_v
gauss = np.exp(-grid**2 / (2 * sigma_v**2)) / np.sqrt(2 * np.pi * sigma_v**2)

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, data, title, nb in [(axes[0], v.ravel(), f"one draw ({v.size} components)", 15),
                            (axes[1], pooled.ravel(), f"500 draws ({pooled.size} components)", 60)]:
    ax.hist(data, bins=nb, density=True, color=PALETTE[0], alpha=0.55, label="sampled")
    ax.plot(grid, gauss, ls="--", color=REFERENCE, lw=1.4, label=r"$\mathcal{N}(0, k_BT/m)$")
    ax.set_title(title)
    ax.set_xlabel(r"velocity component $v_{i,\alpha}$")
axes[0].set_ylabel("probability density")
axes[1].legend()
fig.tight_layout()
plt.show()

## 8. Removing centre-of-mass momentum

A random draw carries a small net momentum, which would make the whole system drift. We subtract

$$\mathbf V_\mathrm{COM} = \frac{\sum_i m_i\mathbf v_i}{\sum_i m_i}, \qquad \mathbf v_i \leftarrow \mathbf v_i - \mathbf V_\mathrm{COM}.$$

In [ ]:
print("total momentum before:", total_momentum(v, mass))
v = remove_com_velocity(v, mass)
print("total momentum after: ", total_momentum(v, mass))

## 9. Instantaneous kinetic temperature

$$K = \sum_i \tfrac12 m_i|\mathbf v_i|^2, \qquad T_\mathrm{inst} = \frac{2K}{f\,k_B}.$$

For $N$ unconstrained particles in $d$ dimensions with the overall translation removed, $f = dN - d$.
In realistic MD, bond constraints (e.g. SHAKE/LINCS) remove further degrees of freedom and must be subtracted from $f$.

In [ ]:
dof = degrees_of_freedom(N, dim)
T_inst = instantaneous_temperature(v, mass, dof)
print(f"f = {dof},  K = {kinetic_energy(v, mass):.3f},  T_inst = {T_inst:.3f}  (target {T_target})")

## 10. Initial temperature rescaling

Multiplying every velocity by
$$\lambda = \sqrt{T_\mathrm{target}/T_\mathrm{inst}}$$
sets the instantaneous kinetic temperature exactly to the target (the momentum stays zero).

In [ ]:
v_scaled, lam = rescale_to_temperature(v, mass, T_target, dof)
print(f"lambda = {lam:.4f}")
print(f"T_inst after rescaling = {instantaneous_temperature(v_scaled, mass, dof):.6f}")
print("total momentum:", total_momentum(v_scaled, mass))

What does this do to the *statistics*? For $N$ = 36 particles, $T_\mathrm{inst}$ of a genuine Maxwell–Boltzmann
draw fluctuates with relative standard deviation $\sqrt{2/f}$. After exact rescaling the fluctuation is gone.

In [ ]:
T_draws = np.array([
    instantaneous_temperature(
        remove_com_velocity(maxwell_boltzmann_velocities(mass, T_target, N, dim, rng), mass), mass, dof)
    for _ in range(2000)
])
print(f"Maxwell-Boltzmann draws: std(T_inst)/T = {T_draws.std() / T_target:.3f}"
      f"   (expected sqrt(2/f) = {np.sqrt(2 / dof):.3f})")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(T_draws, bins=40, density=True, color=PALETTE[0], alpha=0.55, label="Maxwell–Boltzmann draws")
ax.axvline(T_target, color=PALETTE[1], lw=2.5, label="after exact rescaling (every draw)")
ax.set_xlabel(r"$T_\mathrm{inst}$")
ax.set_ylabel("probability density")
ax.legend()
fig.tight_layout()
plt.show()

- Maxwell–Boltzmann sampling produces natural kinetic-energy fluctuations.
- Exact global rescaling pins $T_\mathrm{inst}$, and therefore removes those fluctuations.
- That is acceptable as a one-time **initialization** step: the dynamics (with a proper thermostat) regenerate
  fluctuations afterwards. Repeating the exact rescaling every step is *not* a rigorous canonical thermostat — see Notebook 01.

## Takeaways

- An integrator is a discrete map; its accuracy and stability depend on $\Delta t$ relative to the fastest motion.
- Explicit Euler drifts in energy; Leapfrog and Velocity Verlet keep a bounded energy error at stable timesteps (and give identical positions).
- Initial velocities are drawn from Maxwell–Boltzmann, with centre-of-mass momentum removed and $f = dN - d$ degrees of freedom.
- Exact velocity rescaling fixes $T_\mathrm{inst}$ and suppresses natural fluctuations: use it for initialization, not as a thermostat.